In [17]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

True

In [18]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-06 12:48:11 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: /home/eleni/panellinies/panellinies_exams_dataset/data
2026-04-06 12:48:11 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [19]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-06 12:48:15 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1318 ερωτήσεις-απαντήσεις!


In [11]:
subject_translation = {
    "nea_ellinika": "greek_language",
    "arxaia": "ancient_greek",
    "istoria": "history",
    "latinika": "latin",
    "biologia": "biology",
    "fysiki": "physics",
    "ximeia": "chemistry",
    "pliroforiki": "computer_science",
    "arxes_oikonomikis_theorias": "economics",
    "mathimatika": "mathematics"
}

In [12]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    old_subj = item.get("subject", "")
    new_subj = subject_translation.get(old_subj, old_subj)
    item["subject"] = new_subj
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)
    
    year = item.get("year", "")
    old_id = item.get("id", "")
    school_type = str(item.get("school_type", "gel")).lower()
    item["id"] = f"{new_subj}_{school_type}_{year}_{old_id}"

In [13]:
images_found = 0
print("--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} στο μάθημα {item.get('subject')} ({item.get('year')}) - Περιέχει {len(imgs)} εικόνα/ες")

print(f"\nΣυνολικά βρέθηκαν {images_found} ερωτήσεις (IDs) με εικόνες.")

--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---
ID: physics_gel_2020_A3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B1.α στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B1.β στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B3.α στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_B3.β στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ1 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ2 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Γ4 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ1 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ2 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ3 στο μάθημα physics (2020) - Περιέχει 1 εικόνα/ες
ID: physics_gel_2020_Δ4 στο μάθημα physics (2020) - Περιέχει 1 εικόν

In [14]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "panellinies_dataset.xlsx"

In [15]:
df = pd.DataFrame(main_dataset)
df = df.rename (columns={"answer": "answer_text"})
my_columns = [
    "id",
    "subject",
    "format",
    "reference",
    "question",
    "input",
    "images",
    "choices",
    "answer_text",
    "answer_index",
    "image_description",
    "image_transcription",
    "points",
    "year",
    "school_type"
]

df = df[my_columns]

In [16]:
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

IllegalCharacterError: Εγκάρσιο αρμονικό κύμα, πλάτους $A$ και μήκους κύματος $\lambda$, διαδίδεται χωρίς απώλειες ενέργειας σε ομογενές γραμμικό ελαστικό μέσο μεγάλου μήκους που ταυτίζεται με τον οριζόντιο ημιάξονα $Ox$ προς τη θετική κατεύθυνση, όπως φαίνεται στο σχήμα.
Το κύμα παράγεται από πηγή που βρίσκεται στο σημείο $O$ στη θέση $x=0$ του ελαστικού μέσου και το οποίο αρχίζει να ταλαντώνεται με θετική ταχύτητα τη χρονική στιγμή $t=0$ σύμφωνα με την εξίσωση $y = A \cdot \eta\mu \omega t$.
Το υλικό σημείο $O$ κατά τη διάρκεια της ταλάντωσής του διέρχεται 60 φορές το λεπτό από τη θέση ισορροπίας του.
Κάποια χρονική στιγμή που το υλικό σημείο $O$ βρίσκεται στην ακραία αρνητική του απομάκρυνση ($y = -A$) από την αρχική θέση ισορροπίας του, το υλικό σημείο $\Delta$ του ημιάξονα $Ox$ που απέχει από την πηγή $O$ οριζόντια απόσταση $x_\Delta = 2,5 \text{ m}$ και έχει ήδη αρχίσει να ταλαντώνεται, βρίσκεται στην ακραία θετική του απομάκρυνση ($y = +A$) από την αρχική θέση ισορροπίας του. Τη